# 📘 学习注释版：Gold Product Dimension

**目的：把 CRM Product 与 ERP Product Category 合成商品维表。**

输出：`workspace.gold.dim_products`

重点：
- Silver 多源数据 JOIN
- surrogate key
- Dimension 描述“商品是什么”


#The Transformation Logic

## 🧩 学习说明：构建 Product Dimension

**Input：**
- `silver.crm_products`
- `silver.erp_product_category`

**Process：** JOIN 商品主数据与分类数据，生成 `product_key`

**Output：** 下一步写成 `gold.dim_products`


In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY pn.start_date, pn.product_number) AS product_key, -- Surrogate key
    pn.product_id,
    pn.product_number,
    pn.product_name,
    pn.category_id,
    pc.category,
    pc.subcategory,
    pc.maintenance_flag,
    pn.product_line,
    pn.start_date
FROM silver.crm_products pn
LEFT JOIN silver.erp_product_category pc
    ON pn.category_id = pc.category_id
--WHERE pn.end_date IS NULL; -- Filter out all historical data
"""
df = spark.sql(query)


## 👀 学习说明：DataFrame Sanity Check

只显示前 10 行，快速确认当前 DataFrame：
- 字段是否正确
- 清洗是否生效
- 数据是否仍然存在

这一步不写表，只是开发时的中间检查。


In [0]:
df.limit(10).display()

#Writing Gold Table

## 💾 学习说明：把 DataFrame 持久化为 Delta Table

**Input：** 当前 `df`  
**Process：**
- `mode("overwrite")`：目标已存在时覆盖
- `format("delta")`：使用 Delta 格式
- `saveAsTable()`：注册为 Catalog Table

**Output：** `workspace.gold.dim_products`

注意：这也是为什么 Bootcamp 可以重复运行而通常不会因为“表已存在”直接失败。


In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_products")

## Sanity checks of Gold table

## ✅ 学习说明：验证 Product Dimension

确认：
- `product_key` 已生成
- 商品与分类信息 JOIN 成功


In [0]:
%sql
SELECT * FROM workspace.gold.dim_products LIMIT 10